In [1]:
import pathlib as Path
import pandas as pd
import os

import glob

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [13]:
uoe5_nucleus_output_fn = "/data/kelpschdj/iNDI/Production/notebooks/analysis_uoe5/uoe5_nucleus_output.parquet"
df = pd.read_parquet(uoe5_nucleus_output_fn)

df["Batch"] = df["Batch"].astype("Int64").astype(str)  # Int64 first handles any NaN

print(f"Total number of images: {df[['filename', 'Plate_ID']].drop_duplicates().shape[0]}")
print(f"Total number of nuclei (before QC): {len(df)}")

Total number of images: 193651
Total number of nuclei (before QC): 1034047


In [12]:
df_auto = df[df["Source"] == "automated"].copy()
df_man  = df[df["Source"] == "manual"].copy()

print(f"Total number of images: {df_auto[['filename', 'Plate_ID']].drop_duplicates().shape[0]}")
print(f"Total number of nuclei (before QC): {len(df_auto)}")

print(f"Total number of images: {df_man[['filename', 'Plate_ID']].drop_duplicates().shape[0]}")
print(f"Total number of nuclei (before QC): {len(df_man)}")

Total number of images: 118955
Total number of nuclei (before QC): 578201
Total number of images: 74696
Total number of nuclei (before QC): 455846


Filter parentals across plates and look at which feeatures are plate independent or not\
Itensity focused features seem to be plate dependent while shape features much less so! 

In [16]:
FEATS = ["log_area", "log_intensity_mean", "log_intensity_std",
         "log_aspect_ratio", "log_1m_solidity", "eccentricity"]
WELL_KEYS = ["iNDI_Plate_ID", "Well_ID"]
META = [m for m in ["Gene", "Variant", "Genotype", "Gene_Var_Type", "Batch"]
        if m in df_auto.columns]

g = df_auto.groupby(WELL_KEYS)
for c in META:   # metadata must be constant within a well
    assert (g[c].nunique() <= 1).all(), f"{c} varies within a well"

med = g[FEATS].median().add_suffix("_med")
iqr = (g[FEATS].quantile(0.75) - g[FEATS].quantile(0.25)).add_suffix("_iqr")
wells = pd.concat([g[META].first(), g.size().rename("n_nuclei"), med, iqr], axis=1).reset_index()
wells["log_n_nuclei"] = np.log(wells["n_nuclei"])
wells["is_parental"] = wells["Gene"] == "KOLF2.1J"

WF = [c for c in wells.columns if c.endswith(("_med", "_iqr"))] + ["log_n_nuclei"]
print(wells.shape, len(WF), "features")
print(wells.groupby("iNDI_Plate_ID")["is_parental"].agg(["sum", "size"]))

rng = np.random.default_rng(0)
par = wells[wells["is_parental"]].copy()
# 0 = fit half, 1 = held-out evaluation half (balanced within each plate)
par["half"] = par.groupby("iNDI_Plate_ID")["Well_ID"].transform(
    lambda s: rng.permutation(len(s)) % 2)

def omega_sq(values, groups):
    d = pd.DataFrame({"v": values, "g": groups}).dropna()
    k, N, gm = d["g"].nunique(), len(d), d["v"].mean()
    ssb = d.groupby("g")["v"].apply(lambda s: len(s) * (s.mean() - gm) ** 2).sum()
    sst = ((d["v"] - gm) ** 2).sum()
    msw = (sst - ssb) / (N - k)
    return max(0.0, (ssb - (k - 1) * msw) / (sst + msw))

ev = par[par["half"] == 1]
plate_omega = pd.Series({f: omega_sq(ev[f], ev["iNDI_Plate_ID"]) for f in WF}, name="M0_plate_omega2")
print(plate_omega.sort_values(ascending=False).round(3))

(3740, 22) 13 features
               sum  size
iNDI_Plate_ID           
INDI00002D      96   384
INDI00007D      48   287
INDI00011D      96   191
INDI00015D      48   336
INDI00019D      95   236
INDI00023D      96   239
INDI00026D      93   237
INDI00035D      96   384
INDI00039D      96   288
INDI00043D      96   286
INDI00047D      95   239
INDI00049D      91   352
INDI00053D      94   281
log_intensity_mean_med    0.549
log_intensity_std_med     0.525
log_n_nuclei              0.313
log_1m_solidity_med       0.279
log_intensity_mean_iqr    0.265
log_intensity_std_iqr     0.239
log_area_med              0.204
log_1m_solidity_iqr       0.124
log_aspect_ratio_med      0.090
eccentricity_med          0.085
log_area_iqr              0.066
eccentricity_iqr          0.054
log_aspect_ratio_iqr      0.027
Name: M0_plate_omega2, dtype: float64


Method 1: per plate robust z-score

In [17]:
def robust_z(wells, par_fit, feats, plate_col="iNDI_Plate_ID", eps=1e-9):
    med = par_fit.groupby(plate_col)[feats].median()
    mad = par_fit.groupby(plate_col)[feats].apply(
        lambda d: 1.4826 * (d - d.median()).abs().median())
    out = wells.copy()
    plates = wells[plate_col]
    out[feats] = (wells[feats].to_numpy() - med.loc[plates].to_numpy()) \
                 / (mad.loc[plates].to_numpy() + eps)
    return out

m1 = robust_z(wells, par[par["half"] == 0], WF)
ev1 = m1.loc[ev.index]    # same held-out parental wells as before

m1_omega = pd.Series({f: omega_sq(ev1[f], ev1["iNDI_Plate_ID"]) for f in WF}, name="M1_plate_omega2")
cmp = pd.concat([plate_omega, m1_omega], axis=1).sort_values("M0_plate_omega2", ascending=False)
print(cmp.round(3))

                        M0_plate_omega2  M1_plate_omega2
log_intensity_mean_med            0.549            0.019
log_intensity_std_med             0.525            0.027
log_n_nuclei                      0.313            0.042
log_1m_solidity_med               0.279            0.015
log_intensity_mean_iqr            0.265            0.018
log_intensity_std_iqr             0.239            0.002
log_area_med                      0.204            0.024
log_1m_solidity_iqr               0.124            0.008
log_aspect_ratio_med              0.090            0.030
eccentricity_med                  0.085            0.029
log_area_iqr                      0.066            0.037
eccentricity_iqr                  0.054            0.047
log_aspect_ratio_iqr              0.027            0.000


In [18]:
def eval_method(correct_fn, seeds=range(20)):
    rows = []
    for s in seeds:
        r = np.random.default_rng(s)
        p = wells[wells["is_parental"]].copy()
        p["half"] = p.groupby("iNDI_Plate_ID")["Well_ID"].transform(
            lambda x: r.permutation(len(x)) % 2)
        corr = correct_fn(wells, p[p["half"] == 0], WF)
        e = corr.loc[p.index[p["half"] == 1]]
        obs  = pd.Series({f: omega_sq(e[f], e["iNDI_Plate_ID"]) for f in WF})
        null = pd.Series({f: omega_sq(e[f], r.permutation(e["iNDI_Plate_ID"].to_numpy())) for f in WF})
        rows.append(pd.DataFrame({"obs": obs, "null": null, "seed": s}))
    return pd.concat(rows)

res_M0 = eval_method(lambda w, pf, f: w)
res_M1 = eval_method(robust_z)
summ = pd.concat({"M0": res_M0.groupby(level=0)[["obs", "null"]].mean(),
                  "M1": res_M1.groupby(level=0)[["obs", "null"]].mean()}, axis=1)
print(summ.round(3))

                           M0            M1       
                          obs   null    obs   null
eccentricity_iqr        0.072  0.002  0.025  0.002
eccentricity_med        0.103  0.002  0.038  0.001
log_1m_solidity_iqr     0.126  0.005  0.033  0.004
log_1m_solidity_med     0.278  0.002  0.034  0.001
log_area_iqr            0.058  0.005  0.025  0.005
log_area_med            0.220  0.003  0.024  0.003
log_aspect_ratio_iqr    0.067  0.003  0.025  0.004
log_aspect_ratio_med    0.109  0.003  0.037  0.004
log_intensity_mean_iqr  0.245  0.002  0.024  0.002
log_intensity_mean_med  0.504  0.004  0.011  0.005
log_intensity_std_iqr   0.208  0.002  0.024  0.001
log_intensity_std_med   0.486  0.002  0.023  0.003
log_n_nuclei            0.309  0.003  0.041  0.002


In [19]:
def robust_z_pooled_scale(wells, par_fit, feats, plate_col="iNDI_Plate_ID", eps=1e-9):
    med = par_fit.groupby(plate_col)[feats].median()
    centered = par_fit[feats].to_numpy() - med.loc[par_fit[plate_col]].to_numpy()
    scale = 1.4826 * np.median(np.abs(centered - np.median(centered, axis=0)), axis=0)
    out = wells.copy()
    out[feats] = (wells[feats].to_numpy() - med.loc[wells[plate_col]].to_numpy()) / (scale + eps)
    return out

res_M1b = eval_method(robust_z_pooled_scale)
summ_b = res_M1b.groupby(level=0)[["obs", "null"]].mean()
print(pd.concat({"M1": res_M1.groupby(level=0)["obs"].mean(),
                 "M1b": summ_b["obs"]}, axis=1).round(3))

                           M1    M1b
eccentricity_iqr        0.025  0.030
eccentricity_med        0.038  0.025
log_1m_solidity_iqr     0.033  0.026
log_1m_solidity_med     0.034  0.032
log_area_iqr            0.025  0.021
log_area_med            0.024  0.022
log_aspect_ratio_iqr    0.025  0.036
log_aspect_ratio_med    0.037  0.025
log_intensity_mean_iqr  0.024  0.029
log_intensity_mean_med  0.011  0.014
log_intensity_std_iqr   0.024  0.028
log_intensity_std_med   0.023  0.022
log_n_nuclei            0.041  0.049


In [20]:
vp = (wells[~wells["is_parental"]]
      .groupby("Gene_Var_Type")["iNDI_Plate_ID"].nunique()
      .value_counts().sort_index())
print(vp)    # how many variants are on 1, 2, 3... plates

per_variant = wells[~wells["is_parental"]].groupby("Gene_Var_Type").agg(
    plates=("iNDI_Plate_ID", "nunique"), wells=("Well_ID", "size"))
print(per_variant.sort_values("plates", ascending=False).head(15))

iNDI_Plate_ID
1    52
2     1
Name: count, dtype: int64
                    plates  wells
Gene_Var_Type                    
APOE_R158C_SNV/SNV       2     89
APOE_C112R_SNV/SNV       1     93
APP_A673T_REV/WT         1     48
APP_A673T_SNV/SNV        1     48
APP_A673T_SNV/WT         1     48
APP_E693G_REV/WT         1     48
APP_E693G_SNV/SNV        1     48
APP_E693G_SNV/WT         1     48
APP_V717I_REV/WT         1     48
APP_V717I_SNV/SNV        1     48
APP_V717I_SNV/WT         1     48
FUS_R216C_REV/WT         1     47
FUS_R216C_SNV/SNV        1     47
FUS_R216C_SNV/WT         1     47
FUS_R495X_REV/WT         1     48


In [21]:
from scipy.stats import mannwhitneyu

# final fit: use ALL parental wells (no holdout), pooled scale
corr = robust_z_pooled_scale(wells, wells[wells["is_parental"]], WF)

rows = []
for (gvt, plate), v in corr[~corr["is_parental"]].groupby(["Gene_Var_Type", "iNDI_Plate_ID"]):
    p_ = corr[corr["is_parental"] & (corr["iNDI_Plate_ID"] == plate)]
    for f in WF:
        rows.append(dict(
            Gene_Var_Type=gvt, plate=plate, feature=f,
            effect=v[f].median() - p_[f].median(),          # in pooled robust SD units
            p=mannwhitneyu(v[f], p_[f]).pvalue,
            n_var=len(v), n_par=len(p_)))
eff = pd.DataFrame(rows)
eff["is_rev"] = eff["Gene_Var_Type"].str.contains("REV/WT")

# negative-control spread: how large are |effects| for REV/WT vs everything else?
print(eff.groupby("is_rev")["effect"].apply(lambda s: s.abs().describe()).unstack().round(2))
rev_null = eff[eff["is_rev"]].groupby("feature")["effect"].apply(lambda s: s.abs().quantile(0.95))
print(rev_null.round(2))

        count  mean   std  min   25%   50%   75%   max
is_rev                                                
False   481.0  0.55  0.62  0.0  0.18  0.39  0.69  7.54
True    221.0  0.57  0.72  0.0  0.18  0.35  0.73  6.77
feature
eccentricity_iqr          0.85
eccentricity_med          0.48
log_1m_solidity_iqr       0.93
log_1m_solidity_med       0.99
log_area_iqr              1.84
log_area_med              1.38
log_aspect_ratio_iqr      1.04
log_aspect_ratio_med      0.48
log_intensity_mean_iqr    1.83
log_intensity_mean_med    1.37
log_intensity_std_iqr     1.73
log_intensity_std_med     1.60
log_n_nuclei              4.51
Name: effect, dtype: float64


In [22]:
rev = eff[eff["is_rev"]]
non = eff[~eff["is_rev"]].copy()

# robust spread of revertant effects per feature
rev_sd = rev.groupby("feature")["effect"].apply(
    lambda s: 1.4826 * (s - s.median()).abs().median())
eff["z_rev"] = eff["effect"] / eff["feature"].map(rev_sd)

def mw(f):
    return mannwhitneyu(non.loc[non.feature == f, "effect"].abs(),
                        rev.loc[rev.feature == f, "effect"].abs()).pvalue

feat_summary = pd.DataFrame({
    "rev_sd": rev_sd,
    "rev_p95": rev_null,
    "frac_nonrev_beyond_p95": {f: (non.loc[non.feature == f, "effect"].abs() > rev_null[f]).mean()
                               for f in rev_null.index},
    "mw_p_|effect|": {f: mw(f) for f in rev_null.index},
}).round(3)
print(feat_summary)

top = (eff[~eff["is_rev"]].assign(abs_z=lambda d: d["z_rev"].abs())
       .sort_values("abs_z", ascending=False)
       [["Gene_Var_Type", "plate", "feature", "effect", "z_rev"]].head(25))
print(top.round(2))

                        rev_sd  rev_p95  frac_nonrev_beyond_p95  mw_p_|effect|
eccentricity_iqr         0.480    0.852                   0.081          0.068
eccentricity_med         0.358    0.479                   0.351          0.035
log_1m_solidity_iqr      0.295    0.934                   0.135          0.342
log_1m_solidity_med      0.533    0.991                   0.081          0.539
log_area_iqr             0.710    1.837                   0.027          0.157
log_area_med             0.792    1.381                   0.054          0.682
log_aspect_ratio_iqr     0.262    1.037                   0.000          0.926
log_aspect_ratio_med     0.340    0.481                   0.351          0.035
log_intensity_mean_iqr   0.653    1.834                   0.027          0.737
log_intensity_mean_med   0.632    1.374                   0.081          0.766
log_intensity_std_iqr    0.935    1.728                   0.054          0.926
log_intensity_std_med    0.625    1.602             

In [24]:
eff["abs_z"] = eff["z_rev"].abs()
print(eff.groupby("is_rev")["abs_z"].agg(
    n="size", gt2=lambda s: (s > 2).mean(), gt3=lambda s: (s > 3).mean()).round(3))

by_plate = (eff[~eff["is_rev"]].assign(hit=lambda d: d["abs_z"] > 3)
            .groupby("plate")["hit"].agg(["sum", "size"]))
by_plate["frac"] = by_plate["sum"] / by_plate["size"]
print(by_plate.sort_values("frac", ascending=False).round(2))

def series(prefix, feats):
    d = eff[eff["Gene_Var_Type"].str.startswith(prefix) & eff["feature"].isin(feats)]
    return d.pivot_table(index="Gene_Var_Type", columns="feature", values="effect")

print(series("FUS_R495X",  ["log_intensity_mean_med", "log_intensity_std_med"]).round(2))
print(series("PSEN1_E276A", ["log_n_nuclei", "log_intensity_std_med"]).round(2))
print(series("APP_E693G",  ["log_aspect_ratio_med", "eccentricity_med"]).round(2))

dens = "log_n_nuclei"
adj_feats = [f for f in WF if f != dens]
parc = corr[corr["is_parental"]]
adjusted = corr.copy()
for f in adj_feats:
    slope, intercept = np.polyfit(parc[dens], parc[f], 1)   # fit on parentals only
    adjusted[f] = corr[f] - (slope * corr[dens] + intercept)
# rerun the effect/z_rev calculation on `adjusted` with adj_feats, then compare the top hits

          n    gt2    gt3
is_rev                   
False   481  0.108  0.037
True    221  0.090  0.045
            sum  size  frac
plate                      
INDI00019D    2    13  0.15
INDI00015D    9    65  0.14
INDI00002D    4    52  0.08
INDI00026D    1    26  0.04
INDI00043D    1    39  0.03
INDI00049D    1    52  0.02
INDI00011D    0    26  0.00
INDI00007D    0    39  0.00
INDI00023D    0    26  0.00
INDI00039D    0    26  0.00
INDI00035D    0    52  0.00
INDI00047D    0    26  0.00
INDI00053D    0    39  0.00
feature            log_intensity_mean_med  log_intensity_std_med
Gene_Var_Type                                                   
FUS_R495X_REV/WT                     0.78                   1.05
FUS_R495X_SNV/SNV                    2.80                   2.91
FUS_R495X_SNV/WT                     1.46                   1.52
feature              log_intensity_std_med  log_n_nuclei
Gene_Var_Type                                           
PSEN1_E276A_REV/WT                   

In [25]:
sub = eff[(eff["plate"] == "INDI00002D") &
          eff["feature"].isin(["log_intensity_mean_med", "log_intensity_std_med", "log_n_nuclei"])]
print(sub.pivot_table(index="Gene_Var_Type", columns="feature", values="effect").round(2))
print(series("FUS_R495X", ["log_n_nuclei"]).round(2))

feature            log_intensity_mean_med  log_intensity_std_med  log_n_nuclei
Gene_Var_Type                                                                 
FUS_R495X_REV/WT                     0.78                   1.05         -1.14
FUS_R495X_SNV/SNV                    2.80                   2.91         -0.79
FUS_R495X_SNV/WT                     1.46                   1.52         -0.62
VCP_R159H_REV/WT                     2.18                   3.14         -3.95
VCP_R159H_SNV/SNV                    1.49                   1.13          0.15
VCP_R159H_SNV/WT                    -0.24                  -0.66          0.36
feature            log_n_nuclei
Gene_Var_Type                  
FUS_R495X_REV/WT          -1.14
FUS_R495X_SNV/SNV         -0.79
FUS_R495X_SNV/WT          -0.62


In [26]:
if "JAX_ID" in df_auto.columns:
    print(df_auto.groupby("Gene_Var_Type")["JAX_ID"].nunique().value_counts())

JAX_ID
1    52
2     2
Name: count, dtype: int64


In [27]:
from scipy.stats import binomtest

e = eff.copy()
e[["series", "geno"]] = e["Gene_Var_Type"].str.rsplit("_", n=1, expand=True)
w = e.pivot_table(index=["series", "feature"], columns="geno", values="effect")
w = w.dropna(subset=["REV/WT", "SNV/WT", "SNV/SNV"])

up   = (w["REV/WT"] < w["SNV/WT"]) & (w["SNV/WT"] < w["SNV/SNV"])
down = (w["REV/WT"] > w["SNV/WT"]) & (w["SNV/WT"] > w["SNV/SNV"])
w["mono"] = up | down

rows = []
for f, d in w.groupby(level="feature"):
    k, n = int(d["mono"].sum()), len(d)
    rows.append(dict(feature=f, n_series=n, n_monotone=k, frac=round(k / n, 2),
                     p_vs_1_3=round(binomtest(k, n, 1/3, alternative="greater").pvalue, 3)))
print(pd.DataFrame(rows).sort_values("p_vs_1_3"))

# which variants have more than one clone?
print(df_auto.groupby("Gene_Var_Type")["JAX_ID"].nunique().loc[lambda s: s > 1])

                   feature  n_series  n_monotone  frac  p_vs_1_3
9   log_intensity_mean_med        15           8  0.53     0.088
8   log_intensity_mean_iqr        15           8  0.53     0.088
11   log_intensity_std_med        15           7  0.47     0.203
0         eccentricity_iqr        15           6  0.40     0.382
10   log_intensity_std_iqr        15           6  0.40     0.382
3      log_1m_solidity_med        15           6  0.40     0.382
4             log_area_iqr        15           6  0.40     0.382
5             log_area_med        15           5  0.33     0.596
2      log_1m_solidity_iqr        15           4  0.27     0.791
6     log_aspect_ratio_iqr        15           4  0.27     0.791
7     log_aspect_ratio_med        15           3  0.20     0.921
1         eccentricity_med        15           3  0.20     0.921
12            log_n_nuclei        15           3  0.20     0.921
Gene_Var_Type
APOE_C112R_SNV/SNV    2
APOE_R158C_SNV/SNV    2
Name: JAX_ID, dtype: int64


In [28]:
jax = df_auto.groupby(WELL_KEYS)["JAX_ID"].first().rename("JAX_ID").reset_index()
c2 = corr.merge(jax, on=WELL_KEYS, how="left")
apoe = c2[c2["Gene_Var_Type"].str.startswith("APOE") & ~c2["is_parental"]]

rows = []
for (gvt, jid, plate), v in apoe.groupby(["Gene_Var_Type", "JAX_ID", "iNDI_Plate_ID"]):
    p_ = c2[c2["is_parental"] & (c2["iNDI_Plate_ID"] == plate)]
    for f in WF:
        rows.append(dict(gvt=gvt, jax=jid, plate=plate, feature=f,
                         effect=v[f].median() - p_[f].median()))
ap = pd.DataFrame(rows)
print(ap.pivot_table(index="feature", columns=["gvt", "jax", "plate"], values="effect").round(2))

gvt                    APOE_C112R_SNV/SNV             APOE_R158C_SNV/SNV  \
jax                           JIPSC001142 JIPSC001150        JIPSC001146   
plate                          INDI00053D  INDI00053D         INDI00049D   
feature                                                                    
eccentricity_iqr                    -0.24        0.05              -0.30   
eccentricity_med                     0.10       -0.95              -0.46   
log_1m_solidity_iqr                  0.30        0.02              -0.12   
log_1m_solidity_med                 -0.16       -0.16              -0.92   
log_area_iqr                        -0.58        0.18              -0.90   
log_area_med                         0.55       -0.51               0.90   
log_aspect_ratio_iqr                -0.11       -0.37              -0.13   
log_aspect_ratio_med                 0.10       -0.94              -0.45   
log_intensity_mean_iqr              -0.23        1.15              -1.51   
log_intensit